# LoRA Fine-Tuning: Qwen3.5-2B-Base


Практический notebook по **LoRA (Low-Rank Adaptation)** на `Qwen/Qwen3.5-2B-Base`.

Используется та же задача, что и в предыдущем PEFT / Prompt Tuning notebook: бинарная классификация тональности Stanford SST-2 через causal text generation (`positive` / `negative`).

Это позволяет сравнивать методы при максимально близких условиях: одна base model, один dataset, один prompt format и полный validation split.


## Теоретическая часть


### Что такое LoRA


**LoRA — Low-Rank Adaptation** — один из наиболее распространённых методов Parameter-Efficient Fine-Tuning.

Вместо обновления большой матрицы весов модели LoRA:

- оставляет исходную матрицу `W` замороженной;
- добавляет небольшой обучаемый low-rank update;
- обучает только этот update;
- сохраняет результат как небольшой adapter.

Таким образом, pretrained weights не изменяются напрямую.


### От full fine-tuning к LoRA


При full fine-tuning матрица весов $W$ целиком получает gradients и обновляется optimizer.

Полное обновление можно записать как:

$W' = W + \Delta W$

где $\Delta W$ имеет тот же размер, что и исходная матрица $W$.

LoRA предполагает, что полезное обновление можно аппроксимировать произведением двух матриц низкого ранга:

$W' = W + BA$

При этом $W$ остаётся frozen, а обучаются только матрицы $A$ и $B$.


### Математическая идея


Пусть линейный слой содержит матрицу весов

$W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$.

При full fine-tuning обучается полная матрица обновления

$\Delta W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$.

LoRA представляет это обновление как произведение двух матриц:

$\Delta W = BA$,

где

$A \in \mathbb{R}^{r \times d_{\text{in}}}, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r}$,

а rank $r$ выбирается намного меньше $d_{\text{in}}$ и $d_{\text{out}}$:

$r \ll \min(d_{\text{in}}, d_{\text{out}})$.

Forward pass можно записать как

$y = Wx + s\,B(Ax)$,

где $s$ — scaling factor LoRA.

Исходная матрица $W$ остаётся frozen, а $A$ и $B$ являются trainable.


### Почему это уменьшает число trainable parameters


Для обычного linear layer число параметров матрицы весов равно

$N_{\text{full}} = d_{\text{out}} d_{\text{in}}$.

Для LoRA adapter обучаются две low-rank matrices, поэтому

$N_{\text{LoRA}} = r d_{\text{in}} + d_{\text{out}} r = r(d_{\text{in}} + d_{\text{out}})$.

Если $r$ значительно меньше размерностей слоя, экономия получается очень большой.

Например, для квадратной матрицы

$d_{\text{in}} = d_{\text{out}} = 2048$

полная матрица содержит

$N_{\text{full}} = 2048 \times 2048 = 4\,194\,304$

параметра.

При $r=16$:

$N_{\text{LoRA}} = 16 \times 2048 + 2048 \times 16 = 65\,536$.

Отношение числа параметров:

$\frac{N_{\text{full}}}{N_{\text{LoRA}}} = \frac{4\,194\,304}{65\,536} = 64$.

То есть для такой матрицы LoRA обучает примерно в 64 раза меньше параметров.


### Что означает rank `r`


Rank $r$ определяет размер low-rank пространства, через которое LoRA описывает изменение исходной матрицы.

Маленький $r$:

- меньше trainable parameters;
- меньше optimizer state;
- меньше adapter;
- ниже capacity.

Большой $r$:

- больше trainable parameters;
- больше capacity;
- больше memory footprint;
- adapter становится крупнее.

Для первого эксперимента используется

$r = 16$.

Это распространённая отправная точка, но оптимальный rank зависит от задачи и модели.


### `lora_alpha`


LoRA update масштабируется коэффициентом, связанным с параметром `lora_alpha`.

Для классической LoRA:

$s = \frac{\alpha}{r}$,

где:

- $s$ — scaling factor;
- $\alpha$ — `lora_alpha`;
- $r$ — rank.

В notebook:

$r = 16, \qquad \alpha = 32$.

Поэтому:

$s = \frac{32}{16} = 2$.

`lora_alpha` позволяет изменять вклад LoRA adapter без изменения rank.


### `lora_dropout`


`lora_dropout` применяется к LoRA-ветке во время training и работает как regularization.

В notebook используется значение

$p_{\text{dropout}} = 0.05$.

В конфигурации PEFT ему соответствует `lora_dropout=0.05`.

Во время inference dropout отключается автоматически после `model.eval()`.


### Почему LoRA в начале является no-op


Стандартная инициализация PEFT делает LoRA adapter практически нейтральным в момент создания.

Одна из low-rank matrices инициализируется случайно, другая — нулями. Поэтому в начале обучения

$BA \approx 0$

и, следовательно,

$W' = W + BA \approx W$.

То есть модель первоначально ведёт себя почти как исходная pretrained model, а изменение поведения появляется постепенно в процессе обучения.


### Официальная схема LoRA


![LoRA diagram](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/lora_diagram.png)


На схеме Hugging Face видно, что frozen pretrained weight используется вместе с отдельной trainable low-rank веткой.

Источник: [Hugging Face PEFT — LoRA](https://huggingface.co/docs/peft/main/conceptual_guides/lora).


### Где можно применять LoRA


LoRA можно применять к разным линейным преобразованиям Transformer:

- query / key / value projections;
- attention output projection;
- MLP projections;
- отдельным projection layers современных hybrid architectures.

Именно параметр `target_modules` определяет, куда будут внедрены LoRA matrices.


### Почему `target_modules` особенно важен для Qwen3.5


Qwen3.5 использует гибридную архитектуру: в ней чередуются **Gated DeltaNet linear-attention layers** и обычные **full-attention layers**.

Поэтому модель содержит не только классические:

```text
q_proj
k_proj
v_proj
o_proj
```

но и другие linear projections, используемые DeltaNet.

Если указать только `q_proj` и `v_proj`, LoRA затронет лишь часть архитектуры.

В этом notebook используется:

```python
target_modules="all-linear"
```

PEFT автоматически применяет LoRA ко всем подходящим Linear/Conv1D-модулям pretrained model, исключая output layer.

Перед созданием adapter notebook отдельно выводит реальные имена linear modules Qwen3.5, чтобы решение было проверяемым, а не основанным на предположении об архитектуре.


### Почему `all-linear` полезен для дальнейшего QLoRA


Этот выбор делает будущий эксперимент с QLoRA особенно чистым.

Мы сможем оставить практически неизменными:

```text
dataset
prompt format
LoRA rank
LoRA alpha
target_modules
evaluation
```

и изменить главным образом способ загрузки base model:

```text
LoRA:
BF16 base model + LoRA

QLoRA:
4-bit base model + LoRA
```

Так можно будет отдельно увидеть эффект quantization на VRAM, training speed, adapter quality и итоговую accuracy.


### LoRA и Prompt Tuning


Оба подхода относятся к PEFT, но обучаемые параметры находятся в разных местах.

| Метод | Base weights | Trainable parameters | Где они находятся |
|---|---|---|---|
| Prompt Tuning | frozen | virtual prompt embeddings | перед входными embeddings |
| LoRA | frozen | low-rank matrices A/B | внутри выбранных linear layers |
| Full fine-tuning | trainable | почти все weights | по всей модели |

Prompt Tuning обучает особенно мало параметров.

LoRA обычно имеет больше trainable parameters, но позволяет адаптировать внутренние преобразования модели и поэтому часто показывает более устойчивое качество на широком наборе задач.


### LoRA и QLoRA


**QLoRA не является отдельной заменой LoRA.**

QLoRA использует LoRA adapter, но замороженная base model хранится в low-bit quantized representation, обычно 4-bit.

```text
LoRA
────────────────────────
Base weights: BF16/FP16
Adapter:      trainable LoRA


QLoRA
────────────────────────
Base weights: 4-bit
Adapter:      trainable LoRA
```

Именно поэтому перед QLoRA отдельно полезно изучить quantization через `bitsandbytes`.


### Merge LoRA adapter


После обучения LoRA adapter можно использовать двумя способами.

**Отдельный adapter:**

```text
base model + adapter
```

Плюсы:

- маленький файл;
- можно хранить много adapters для одной base model;
- легко переключать задачи.

**Merged model:**

После merge low-rank update добавляется к исходной матрице весов:

$W_{\text{merged}} = W + \Delta W = W + BA$.

PEFT предоставляет `merge_and_unload()`, который выполняет это объединение.

Это удобно для deployment, но после merge нужно сохранять уже полную модель, поэтому размер артефакта резко увеличивается.


### Современные расширения LoRA


Классическая LoRA остаётся базовой точкой отсчёта, но современные версии PEFT поддерживают несколько расширений, которые меняют scaling, decomposition или initialization.

В этом notebook они **не включаются**, чтобы эксперимент оставался чистым примером классической LoRA. Но понимать их важно перед переходом к более сложным fine-tuning сценариям.


### rsLoRA


**Rank-Stabilized LoRA (rsLoRA)** меняет scaling low-rank update.

Классическая LoRA использует

$s_{\text{LoRA}} = \frac{\alpha}{r}$.

rsLoRA использует

$s_{\text{rsLoRA}} = \frac{\alpha}{\sqrt{r}}$.

При росте rank $r$ scaling rsLoRA уменьшается медленнее, чем в классической LoRA, что помогает стабильнее использовать большие rank.

В PEFT включается через:

```python
use_rslora=True
```


### DoRA


**DoRA — Weight-Decomposed Low-Rank Adaptation** — разделяет изменение веса на magnitude и direction.

LoRA в основном моделирует low-rank изменение направления весов, а DoRA дополнительно обучает magnitude-компоненту.

Это может улучшать качество, особенно при небольших rank, но увеличивает вычислительную сложность и число trainable parameters.


### PiSSA


**PiSSA — Principal Singular values and Singular vectors Adaptation** использует SVD-based initialization.

Вместо стандартной случайной инициализации LoRA matrices адаптер инициализируется на основе ведущих singular components исходного веса.

Цель — начать обучение из более информативного low-rank пространства и ускорить адаптацию.


### EVA


**EVA — Explained Variance Adaptation** использует активации на реальных данных, чтобы определить более полезное low-rank subspace.

То есть initialization учитывает не только сами веса модели, но и то, какие направления реально активируются на данных конкретной задачи.


### LoftQ


**LoftQ** предназначен прежде всего для quantized fine-tuning.

Он совместно учитывает quantization error и LoRA initialization, чтобы low-rank adapter компенсировал часть ошибки, возникающей после квантования base weights.

Поэтому LoftQ особенно логично рассматривать уже вместе с QLoRA.


### CorDA


**CorDA — Context-Oriented Decomposition Adaptation** строит low-rank decomposition с учётом контекста и статистики данных.

Идея заключается в том, чтобы направить адаптацию в компоненты пространства весов, наиболее полезные для downstream task, сохраняя при этом знания base model.


### Почему в эксперименте остаётся классическая LoRA


В текущем notebook используется:

```python
use_rslora=False
```

и стандартная LoRA initialization.

Это нужно для чистой baseline-точки:

```text
Prompt Tuning
↓
Classic LoRA
↓
Quantization
↓
QLoRA
↓
современные LoRA-варианты
```

Если сразу включить rsLoRA, PiSSA или DoRA, станет сложнее понять, какое именно изменение повлияло на качество.


### `torch.compile` и LoRA


`torch.compile` — это не PEFT-метод, а механизм PyTorch compilation.

Transformers может компилировать training graph через параметр `torch_compile=True` в `TrainingArguments`.

Потенциальная выгода:

- ускорение повторяющихся forward/backward passes;
- оптимизация graph execution;
- особенно заметный эффект на более длинных training runs.

Цена:

- первый запуск требует времени на compilation;
- для короткого smoke-run compilation может не окупиться;
- некоторые модели и операции могут приводить к graph breaks.

Поэтому notebook содержит переключатель:

```python
USE_TORCH_COMPILE = False
```

Он отключён по умолчанию и может быть включён отдельно для измерения реального эффекта.


### Что будем измерять


Практическая часть проверяет LoRA не только функционально, но и количественно:

1. параметры исходной Qwen3.5;
2. список реальных Linear modules;
3. количество LoRA-target modules;
4. число trainable parameters;
5. долю trainable parameters;
6. baseline accuracy на полном SST-2 validation;
7. LoRA accuracy на том же полном validation;
8. valid label output rate;
9. размер LoRA adapter;
10. возможность clean reload;
11. возможность merge adapter в base model;
12. создание полноценной Hugging Face Model Card.


### Теоретические источники


- LoRA paper: https://arxiv.org/abs/2106.09685
- Hugging Face PEFT LoRA conceptual guide: https://huggingface.co/docs/peft/main/conceptual_guides/lora
- Hugging Face PEFT LoRA API: https://huggingface.co/docs/peft/main/package_reference/lora
- Hugging Face Transformers PEFT integration: https://huggingface.co/docs/transformers/peft
- Qwen3.5 documentation: https://huggingface.co/docs/transformers/model_doc/qwen3_5
- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base


## Практическая часть


### 1. Импорты и проверка среды


Зависимости должны устанавливаться в Docker image, а не внутри notebook.

Notebook только проверяет, что версии библиотек достаточно новые для используемого API. Стандартные classification metrics рассчитываются через `TorchMetrics`, чтобы evaluation оставался внутри PyTorch-стека.

In [3]:
import gc
import sys
from collections import Counter, defaultdict
from contextlib import contextmanager
from dataclasses import dataclass
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any

import numpy as np
import plotly.graph_objects as go
import torch
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi, create_repo
from packaging.version import Version
from peft import LoraConfig, PeftConfig, PeftModel, TaskType, get_peft_model
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, EarlyStoppingCallback, Trainer, TrainingArguments, set_seed

print(f"Python:           {sys.version.split()[0]}")
print(f"PyTorch:          {torch.__version__}")
print(f"Transformers:     {package_version('transformers')}")
print(f"PEFT:             {package_version('peft')}")
print(f"Datasets:         {package_version('datasets')}")
print(f"Accelerate:       {package_version('accelerate')}")
print(f"TorchMetrics:     {package_version('torchmetrics')}")
print(f"Hugging Face Hub: {package_version('huggingface_hub')}")

print(f"\nCUDA available:   {torch.cuda.is_available()}")
print(f"Plotly:           {package_version('plotly')}")


Python:           3.10.12
PyTorch:          2.13.0+cu130
Transformers:     5.14.1
PEFT:             0.20.0
Datasets:         5.0.1
Accelerate:       1.14.0
TorchMetrics:     1.9.0
Hugging Face Hub: 1.28.0

CUDA available:   True
Plotly:           6.9.0


In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


In [5]:
def clear_device_memory():
    """Release Python garbage and unused CUDA allocator cache."""
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


### 2. Конфигурация


Notebook поддерживает два режима:

```text
RUN_MODE = "smoke"  → 8 000 training examples
RUN_MODE = "full"   → весь SST-2 train split
```

`smoke` используется по умолчанию и рассчитан на запуск всего notebook через **Run All**. В нём используется больше уникальных training examples, чем в первоначальном варианте, чтобы получить больше optimizer steps без лишнего повторения одних и тех же данных дополнительными эпохами.

При текущем effective batch size:

$B_{\text{effective}} = 8 \times 2 = 16$

Для 8 000 training examples одна эпоха содержит примерно

$\frac{8000}{16} = 500$

optimizer steps, а три эпохи дают примерно

$500 \times 3 = 1500$

optimizer steps.

Validation split всегда остаётся полным.

`USE_GRADIENT_CHECKPOINTING=False` выбран по умолчанию: для 2B-модели checkpointing не нужен, если модель помещается в память. Его стоит включать при переходе к более крупной модели или batch.

`USE_TORCH_COMPILE=False` также является безопасным default. Для длительного training run его можно включить и сравнить фактическое время обучения.

Hugging Face cache используется стандартный. Локально сохраняются checkpoints, reproducibility metadata и финальный LoRA adapter. Training dynamics отображается только внутри notebook.


In [6]:
SEED = 42

MODEL_ID = "Qwen/Qwen3.5-2B-Base"
DATASET_ID = "stanfordnlp/sst2"

MODEL_REVISION = None
DATASET_REVISION = None

TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"
LABEL_NAMES = {
    0: "negative",
    1: "positive",
}

RUN_MODE = "smoke"
assert RUN_MODE in {"smoke", "full"}

MAX_LENGTH = 128
MAX_TRAIN_SAMPLES = 8_000 if RUN_MODE == "smoke" else None
MAX_EVAL_SAMPLES = None

BASELINE_EVAL_SAMPLES = None
FINAL_EVAL_SAMPLES = None
GENERATION_BATCH_SIZE = 32
FORCED_CHOICE_BATCH_SIZE = 16
MAX_NEW_TOKENS = 4

ARTIFACT_RELOAD_SAMPLES = 8
RUN_ARTIFACT_RELOAD_TEST = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 1e-5
LR_SCHEDULER_TYPE = "linear"
WARMUP_STEPS = 0.05
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0
EVAL_STEPS = 100
SAVE_STEPS = EVAL_STEPS
WEIGHT_DECAY = 0.0
USE_GRADIENT_CHECKPOINTING = False
USE_TORCH_COMPILE = False

PUSH_TO_HUB = False
HUB_MODEL_ID = "artyomboyko/qwen3.5-2b-sst2-lora"

cwd = Path.cwd().resolve()

if cwd.name == "peft":
    NOTEBOOK_DIR = cwd
elif (cwd / "notebooks" / "finetuning" / "peft").is_dir():
    NOTEBOOK_DIR = (cwd / "notebooks" / "finetuning" / "peft").resolve()
elif Path("/workspace/notebooks/finetuning/peft").is_dir():
    NOTEBOOK_DIR = Path("/workspace/notebooks/finetuning/peft")
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "qwen3.5-2b-sst2-lora"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

print(f"Notebook directory:   {NOTEBOOK_DIR}")
print(f"Output directory:     {OUTPUT_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

RESULTS_TABLE = []


Notebook directory:   /workspace/notebooks/finetuning/peft
Output directory:     /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-lora
Checkpoint directory: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-lora/checkpoints


### 3. Загрузка SST-2


Используется Stanford SST-2 с двумя классами: `negative` и `positive`.

Validation split сохраняется полностью. Notebook проверяет это assertion, чтобы случайно не получить оценку только на подмножестве.


In [7]:
raw_dataset = load_dataset(
    DATASET_ID,
    revision=DATASET_REVISION,
)

dataset = DatasetDict(
    train=raw_dataset["train"],
    validation=raw_dataset["validation"],
)

def limit_split(split, max_samples):
    if max_samples is None:
        return split

    count = min(max_samples, len(split))
    return split.shuffle(seed=SEED).select(range(count))

dataset["train"] = limit_split(
    dataset["train"],
    MAX_TRAIN_SAMPLES,
)

dataset["validation"] = limit_split(
    dataset["validation"],
    MAX_EVAL_SAMPLES,
)

print(dataset)
print(dataset["train"][0])

print(f"\nTrain samples:      {len(dataset['train']):,}")
print(
    f"Validation samples: {len(dataset['validation']):,} "
    f"/ {len(raw_dataset['validation']):,}"
)

if MAX_EVAL_SAMPLES is None:
    assert len(dataset["validation"]) == len(raw_dataset["validation"]), (
        "Validation split was unexpectedly truncated."
    )

dataset_train_fingerprint = dataset["train"]._fingerprint
dataset_validation_fingerprint = dataset["validation"]._fingerprint

try:
    resolved_dataset_revision = (
        DATASET_REVISION
        or HfApi().dataset_info(DATASET_ID).sha
    )
except Exception:
    resolved_dataset_revision = (
        DATASET_REVISION
        or "unavailable"
    )

print(f"Dataset revision:   {resolved_dataset_revision}")
print(f"Train fingerprint:  {dataset_train_fingerprint}")
print(f"Valid fingerprint:  {dataset_validation_fingerprint}")


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
})
{'idx': 32326, 'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1}

Train samples:      8,000
Validation samples: 872 / 872
Dataset revision:   8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb
Train fingerprint:  637fb4ba0a441ab8
Valid fingerprint:  c1ddc6497ec97f98


### 4. Tokenizer и Qwen3.5-2B-Base


Модель загружается без quantization. Это принципиально: данный notebook показывает обычную LoRA.

Low-bit загрузка будет добавлена только в будущем QLoRA notebook.


In [8]:
use_bf16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

model_dtype = (
    torch.bfloat16
    if use_bf16
    else torch.float16
    if device.type == "cuda"
    else torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=model_dtype,
)

base_model = base_model.to(device)

print(f"Loaded class: {type(base_model).__name__}")
print(f"Model dtype:  {next(base_model.parameters()).dtype}")
print(f"Vocabulary:   {len(tokenizer):,}")
print(f"EOS token:    {tokenizer.eos_token!r}")
print(f"PAD token:    {tokenizer.pad_token!r}")

resolved_model_revision = (
    MODEL_REVISION
    or getattr(base_model.config, "_commit_hash", None)
)

if not resolved_model_revision:
    try:
        resolved_model_revision = HfApi().model_info(
            MODEL_ID
        ).sha
    except Exception:
        resolved_model_revision = "unavailable"

print(f"Model revision: {resolved_model_revision}")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded class: Qwen3_5ForCausalLM
Model dtype:  torch.bfloat16
Vocabulary:   248,077
EOS token:    '<|endoftext|>'
PAD token:    '<|endoftext|>'
Model revision: b1485b2fa6dfa1287294f269f5fb618e03d52d7c


### 5. Параметры base model


До LoRA считаем параметры исходной модели и приблизительный размер весов в текущем dtype.


In [9]:
def parameter_stats(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return total, trainable

def parameter_memory_gib(model):
    total_bytes = sum(
        parameter.numel() * parameter.element_size()
        for parameter in model.parameters()
    )
    return total_bytes / (1024 ** 3)

base_total_params, base_trainable_params = parameter_stats(base_model)

print(f"Total parameters:     {base_total_params:,}")
print(f"Trainable parameters: {base_trainable_params:,}")
print(f"Trainable share:      {100 * base_trainable_params / base_total_params:.6f}%")
print(f"Parameter memory:     {parameter_memory_gib(base_model):.2f} GiB")


Total parameters:     1,881,825,088
Trainable parameters: 1,881,825,088
Trainable share:      100.000000%
Parameter memory:     3.51 GiB


### 6. Поиск Linear-модулей Qwen3.5


Перед заданием LoRA targets notebook исследует модель напрямую.

Это особенно важно для Qwen3.5 из-за гибридной Gated DeltaNet / full-attention архитектуры.


In [10]:
linear_modules = [
    (name, module)
    for name, module in base_model.named_modules()
    if isinstance(module, torch.nn.Linear)
]

linear_leaf_counts = Counter()
linear_leaf_parameters = defaultdict(int)

for name, module in linear_modules:
    leaf_name = name.rsplit(".", 1)[-1]
    linear_leaf_counts[leaf_name] += 1
    linear_leaf_parameters[leaf_name] += sum(
        parameter.numel()
        for parameter in module.parameters(
            recurse=False
        )
    )

print(f"Linear modules found: {len(linear_modules)}")
print(f"Unique leaf names:    {len(linear_leaf_counts)}")

print("\nLinear module statistics:")
print(
    f"{'module':28s} "
    f"{'count':>8s} "
    f"{'parameters':>16s}"
)
print("-" * 56)

for name in sorted(linear_leaf_counts):
    print(
        f"{name:28s} "
        f"{linear_leaf_counts[name]:8d} "
        f"{linear_leaf_parameters[name]:16,d}"
    )

print("\nExamples:")
for name, module in linear_modules[:20]:
    print(
        f"{name:75s} "
        f"{module.in_features:5d} -> {module.out_features:5d}"
    )


Linear modules found: 187
Unique leaf names:    13

Linear module statistics:
module                          count       parameters
--------------------------------------------------------
down_proj                          24      301,989,888
gate_proj                          24      301,989,888
in_proj_a                          18          589,824
in_proj_b                          18          589,824
in_proj_qkv                        18      226,492,416
in_proj_z                          18       75,497,472
k_proj                              6        6,291,456
lm_head                             1      508,559,360
o_proj                              6       25,165,824
out_proj                           18       75,497,472
q_proj                              6       50,331,648
up_proj                            24      301,989,888
v_proj                              6        6,291,456

Examples:
model.layers.0.linear_attn.out_proj                                          2048 ->

### 7. Формат задачи


Как и в Prompt Tuning experiment, SST-2 классификация формулируется как causal language modeling.

Loss считается только по токенам целевой метки.


In [11]:
VISIBLE_INSTRUCTION = (
    "Classify the sentiment of this movie review as positive or negative."
)

def build_prompt(text):
    return (
        f"{VISIBLE_INSTRUCTION}\n"
        f"Review: {text.strip()}\n"
        "Sentiment:"
    )

def preprocess_batch(examples):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for text, label_id in zip(
        examples[TEXT_COLUMN],
        examples[LABEL_COLUMN],
    ):
        target_text = " " + LABEL_NAMES[int(label_id)]

        target_ids = tokenizer(
            target_text,
            add_special_tokens=False,
        )["input_ids"]

        target_ids = target_ids + [tokenizer.eos_token_id]

        max_prompt_length = max(
            1,
            MAX_LENGTH - len(target_ids),
        )

        prompt_ids = tokenizer(
            build_prompt(text),
            add_special_tokens=False,
            truncation=True,
            max_length=max_prompt_length,
        )["input_ids"]

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        all_input_ids.append(input_ids)
        all_attention_masks.append([1] * len(input_ids))
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }

processed_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=1_000,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing SST-2",
)

print(processed_dataset)


Tokenizing SST-2:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing SST-2:   0%|          | 0/872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 872
    })
})


### 8. Data collator


Padding выполняется динамически по текущему batch. Padding в labels заменяется на `-100`, поэтому эти позиции не участвуют в loss.


In [12]:
@dataclass
class CausalClassificationCollator:
    tokenizer: Any

    def __call__(self, features):
        model_features = [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            return_tensors="pt",
        )

        sequence_length = batch["input_ids"].shape[1]

        padded_labels = []

        for feature in features:
            labels = feature["labels"]
            padding_length = sequence_length - len(labels)
            padded_labels.append(
                labels + [-100] * padding_length
            )

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch

data_collator = CausalClassificationCollator(
    tokenizer=tokenizer,
)

test_batch = data_collator(
    [
        processed_dataset["train"][0],
        processed_dataset["train"][1],
    ]
)

for key, value in test_batch.items():
    print(f"{key:16s}: {tuple(value.shape)} {value.dtype}")


input_ids       : (2, 37) torch.int64
attention_mask  : (2, 37) torch.int64
labels          : (2, 37) torch.int64


### 9. Метрики и функции оценки


#### Почему одной accuracy недостаточно


SST-2 — бинарная классификация, но в этом notebook она реализована через causal language model. Поэтому ошибка может возникнуть по двум разным причинам:

1. модель неверно определила sentiment;
2. модель определила sentiment правильно, но нарушила ожидаемый формат ответа.

Например, reference label — `positive`, а модель отвечает:

```text
The review is clearly positive.
```

Смысл ответа правильный, но строгий parser может не принять его как контрактную метку.

Поэтому notebook использует два режима оценки — generation-based и forced-choice — и несколько взаимодополняющих метрик.


#### Generation-based accuracy


**Generation-based accuracy** — доля всех validation examples, для которых свободно сгенерированная и распознанная метка совпала с reference.

Формула:

$\mathrm{Accuracy} = \frac{N_{\mathrm{correct}}}{N_{\mathrm{all}}}$

где:

- $N_{\mathrm{correct}}$ — число правильных предсказаний;
- $N_{\mathrm{all}}$ — общее число примеров;
- `invalid`-ответ тоже считается неправильным, потому что модель не выполнила output contract.

Пример. Пусть на 100 примерах получено:

```text
89 правильных ответов
8 неправильных label
3 invalid output
```

Тогда:

$\mathrm{Accuracy} = \frac{89}{100} = 0.89 = 89\%$

Эта метрика отвечает на практический вопрос: **какая доля входов приводит к полностью корректному результату при реальном `model.generate()`?**


#### Valid output rate


**Valid output rate** показывает, какая доля генераций вообще распознаётся как одна из допустимых меток:

```text
negative
positive
```

Формула:

$\mathrm{ValidOutputRate} = \frac{N_{\mathrm{valid}}}{N_{\mathrm{all}}}$

где:

- $N_{\mathrm{valid}}$ — число ответов, которые parser смог преобразовать в допустимый label;
- $N_{\mathrm{all}}$ — общее число примеров.

Пример. Из 100 ответов:

```text
97 удалось распознать как positive/negative
3 ответа оказались invalid
```

Тогда:

$\mathrm{ValidOutputRate} = \frac{97}{100} = 97\%$

Если forced-choice accuracy высокая, а valid output rate низкая, это сильный сигнал, что модель понимает задачу, но нестабильно соблюдает формат ответа.


#### Forced-choice accuracy


При **forced-choice evaluation** модель не генерирует произвольный текст. В общем случае допустимые labels можно сравнивать по conditional log-probability их токенов после prompt.

Для label из токенов $y_1,\dots,y_T$:

$S(y \mid x) = \sum_{t=1}^{T} \log P(y_t \mid x, y_{<t})$

В этом эксперименте ` negative` и ` positive` проверяются tokenizer-ом как **однотокенные labels**. Поэтому сравнение полной conditional log-probability сводится к сравнению logits этих двух токенов в следующей позиции после prompt: softmax для обоих кандидатов имеет один и тот же знаменатель и не меняет порядок значений.

Предсказание:

$\hat y = \arg\max_{y \in \{\text{negative},\text{positive}\}} z_y$

где $z_y$ — next-token logit соответствующего label token.

Такой вариант даёт ту же forced-choice семантику для однотокенных labels, но требует только одного forward pass на каждый prompt batch и не строит отдельные последовательности `prompt + label`.

#### Precision


**Precision** измеряет точность предсказаний конкретного класса.

Для класса `positive`:

$\mathrm{Precision} = \frac{TP}{TP + FP}$

где:

- $TP$ — true positives: модель предсказала `positive`, и reference тоже `positive`;
- $FP$ — false positives: модель предсказала `positive`, но reference был `negative`.

Пусть:

$TP=43, \qquad FP=3$.

Тогда:

$\mathrm{Precision} = \frac{43}{43+3} = 0.9348 \approx 93.48\%$.

Интерпретация: из всех примеров, которые модель назвала `positive`, примерно 93.5% действительно были positive.

Высокая precision означает мало false positives.


#### Recall


**Recall** измеряет, какую долю реально существующих объектов класса модель смогла обнаружить.

Для класса `positive`:

$\mathrm{Recall} = \frac{TP}{TP + FN}$

где:

- $TP$ — correctly predicted positive examples;
- $FN$ — positive examples, которые модель не классифицировала как `positive`.

В generation-based evaluation в $FN$ попадают и неправильный label, и `invalid` output.

Пример. Пусть среди 50 реально positive examples:

- 43 классифицированы как `positive`;
- 5 — как `negative`;
- 2 дали `invalid` output.

Тогда:

$TP=43, \qquad FN=5+2=7$.

И:

$\mathrm{Recall} = \frac{43}{43+7} = \frac{43}{50} = 86\%$.

Высокий recall означает, что модель редко пропускает реальные объекты данного класса.


#### F1-score


**F1-score** объединяет precision и recall в одну метрику через гармоническое среднее:

$F_1 = 2 \cdot \frac{\mathrm{Precision}\cdot\mathrm{Recall}} {\mathrm{Precision}+\mathrm{Recall}}$.

Гармоническое среднее штрафует сильный дисбаланс между precision и recall сильнее, чем обычное арифметическое среднее.

Пример для `positive`:

$\mathrm{Precision}=0.9348, \qquad \mathrm{Recall}=0.8600$.

Тогда:

$F_1 = 2 \cdot \frac{0.9348\cdot0.86} {0.9348+0.86} \approx 0.8958 = 89.58\%$.

F1 полезен, когда важно одновременно контролировать и false positives, и false negatives.


#### Macro F1


Notebook считает F1 отдельно для `negative` и `positive`, а затем вычисляет **Macro F1** — простое среднее F1 по классам:

$\mathrm{MacroF1} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}$

где:

- $K$ — число классов;
- $F_{1,k}$ — F1 для класса $k$.

Для SST-2:

$K=2$.

Поэтому:

$\mathrm{MacroF1} = \frac{ F_{1,\mathrm{negative}} + F_{1,\mathrm{positive}} }{2}$.

Пусть:

$F_{1,\mathrm{negative}}=0.9110, \qquad F_{1,\mathrm{positive}}=0.8958$.

Тогда:

$\mathrm{MacroF1} = \frac{0.9110+0.8958}{2} = 0.9034 = 90.34\%$.

Macro F1 даёт **одинаковый вес каждому классу**, независимо от количества примеров. Для SST-2 классы достаточно сбалансированы, но метрика всё равно полезна как более подробное дополнение к accuracy.


#### Confusion matrix


**Confusion matrix** — таблица количества предсказаний по комбинации `actual × predicted`.

Для generation-based evaluation notebook использует дополнительный столбец `invalid`:

| Actual \ Predicted | negative | positive | invalid |
|---|---:|---:|---:|
| negative | 46 | 3 | 1 |
| positive | 5 | 43 | 2 |

Для класса `positive` из этой матрицы получаем:

$TP=43, \qquad FP=3, \qquad FN=5+2=7, \qquad TN=46$.

А для `negative` роли классов меняются.

Confusion matrix не сворачивает поведение модели в одну цифру. Она показывает **структуру ошибок**: например, склонна ли модель чаще путать positive с negative или проблема в основном связана с invalid generation.

Для forced-choice evaluation столбца `invalid` нет, потому что алгоритм всегда выбирает один из двух labels.


#### Label Token Accuracy


**Label Token Accuracy** используется как лёгкая training-time метрика и показывает, какая доля целевых label tokens была предсказана правильно.

Она считается только на позициях, участвующих в supervised loss. Prompt и padding в `labels` имеют значение `-100` и в расчёт не входят. EOS отдельно исключается, чтобы метрика отражала именно предсказание содержимого label.

Формула:

$\mathrm{LabelTokenAccuracy} = \frac{N_{\mathrm{correct\ target\ tokens}}}{N_{\mathrm{target\ tokens}}}$

где:

- $N_{\mathrm{correct\ target\ tokens}}$ — число правильно предсказанных токенов label;
- $N_{\mathrm{target\ tokens}}$ — общее число оцениваемых target tokens.

Пример. Если оценивается 100 target tokens и модель правильно предсказала 97:

$\mathrm{LabelTokenAccuracy} = \frac{97}{100} = 97\%$

Эта метрика отличается от generation-based accuracy: она проверяет next-token prediction непосредственно внутри causal-LM evaluation и не требует `model.generate()`.

Поэтому её удобно рассчитывать при промежуточной validation практически без отдельного inference-прохода.


#### Perplexity


**Perplexity (PPL)** — метрика языковой модели, напрямую связанная с average negative log-likelihood.

В notebook она рассчитывается из validation loss:

$\mathrm{PPL} = e^{L_{\mathrm{eval}}}$

где:

- $L_{\mathrm{eval}}$ — causal language modeling loss на validation split;
- $e$ — основание натурального логарифма.

Поскольку prompt positions замаскированы через `-100`, loss относится только к незамаскированной target sequence.

Пример. Если:

$L_{\mathrm{eval}} = 0.093159$

то:

$\mathrm{PPL} = e^{0.093159} \approx 1.098$

Чем меньше perplexity, тем выше средняя уверенность модели в правильных target tokens.

Perplexity не заменяет classification metrics: она не показывает напрямую, правильно ли выбран класс `positive` или `negative`. Для этого используются generation-based accuracy, forced-choice accuracy, precision, recall и F1.


#### Почему используются оба режима


Два режима оценки отвечают на разные вопросы:

```text
Forced-choice
→ понимает ли модель, какой label вероятнее?

Generation-based
→ может ли модель реально выдать этот label в требуемом формате?
```

Их совместная интерпретация особенно полезна:

```text
forced-choice высокий + generation низкий
→ задача в целом понята, проблема скорее в output format

forced-choice высокий + generation высокий
→ модель понимает задачу и стабильно выполняет inference contract

оба низкие
→ fine-tuning ещё не дал достаточного task adaptation
```

Precision, recall, F1 и confusion matrix затем показывают, **какие именно ошибки скрываются внутри общей accuracy**.


Accuracy, precision, recall, F1 и confusion matrix объединены в один `TorchMetrics.MetricCollection`. Generation и forced-choice используют integer class IDs `0/1`, а `invalid` представлен отдельным prediction class `2`.

`Label Token Accuracy` и perplexity остаются специализированными training-time метриками: первая считается напрямую по masked target-token positions, а perplexity — из validation loss. Это позволяет не хранить полный vocabulary-sized logits только ради метрик.

In [13]:
CLASS_LABELS = ("negative", "positive")
INVALID_LABEL_ID = len(CLASS_LABELS)
NUM_EVAL_CLASSES = INVALID_LABEL_ID + 1

EVAL_METRICS = MetricCollection(
    {
        "accuracy": MulticlassAccuracy(NUM_EVAL_CLASSES, average="micro"),
        "precision": MulticlassPrecision(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "recall": MulticlassRecall(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "f1": MulticlassF1Score(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "confusion_matrix": MulticlassConfusionMatrix(
            NUM_EVAL_CLASSES,
        ),
    }
)

_label_token_ids = [
    tokenizer(
        f" {label}",
        add_special_tokens=False,
    )["input_ids"]
    for label in CLASS_LABELS
]

if not all(len(ids) == 1 for ids in _label_token_ids):
    raise ValueError(
        "Forced-choice evaluation requires single-token labels."
    )

LABEL_TOKEN_IDS = torch.tensor(
    [ids[0] for ids in _label_token_ids],
    dtype=torch.long,
)


@contextmanager
def temporary_padding_side(tokenizer, padding_side):
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = padding_side
    try:
        yield
    finally:
        tokenizer.padding_side = original_padding_side


def select_eval_split(raw_split, max_samples):
    count = (
        len(raw_split)
        if max_samples is None
        else min(max_samples, len(raw_split))
    )
    return raw_split.select(range(count))


def tokenize_eval_batch(
    texts,
    device,
    *,
    max_length=MAX_LENGTH,
    add_special_tokens=True,
):
    return tokenizer(
        [build_prompt(text) for text in texts],
        add_special_tokens=add_special_tokens,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)


def normalize_prediction(text):
    text = text.strip().lower()
    return next(
        (
            index
            for index, label in enumerate(CLASS_LABELS)
            if text.startswith(label)
        ),
        INVALID_LABEL_ID,
    )


def classification_metrics(predictions, references):
    predictions = torch.as_tensor(
        predictions,
        dtype=torch.long,
    )
    references = torch.as_tensor(
        references,
        dtype=torch.long,
    )

    values = EVAL_METRICS.clone()(
        predictions,
        references,
    )

    precision = values["precision"][: len(CLASS_LABELS)]
    recall = values["recall"][: len(CLASS_LABELS)]
    f1 = values["f1"][: len(CLASS_LABELS)]

    return {
        "accuracy": values["accuracy"].item(),
        "macro_f1": f1.mean().item(),
        "per_class": {
            label: {
                "precision": precision[index].item(),
                "recall": recall[index].item(),
                "f1": f1[index].item(),
            }
            for index, label in enumerate(CLASS_LABELS)
        },
        "confusion_matrix": (
            values["confusion_matrix"][: len(CLASS_LABELS)]
            .to(torch.int64)
            .tolist()
        ),
        "total": references.numel(),
    }


def print_classification_summary(title, metrics):
    print(f"\n{title}\n{'-' * len(title)}")
    print(f"Accuracy:  {metrics['accuracy']:.2%}")
    print(f"Macro F1:  {metrics['macro_f1']:.4f}")

    for label in CLASS_LABELS:
        values = metrics["per_class"][label]
        print(
            f"{label:8s} "
            f"precision={values['precision']:.4f} "
            f"recall={values['recall']:.4f} "
            f"f1={values['f1']:.4f}"
        )

    if "valid_output_rate" in metrics:
        print(
            "Valid output rate: "
            f"{metrics['valid_output_rate']:.2%}"
        )

    print(
        "\nConfusion matrix "
        "(rows=actual, columns=predicted)"
    )
    print(
        f"{'':12s}"
        f"{'negative':>10s}"
        f"{'positive':>10s}"
        f"{'invalid':>10s}"
    )

    for label, row in zip(
        CLASS_LABELS,
        metrics["confusion_matrix"],
    ):
        print(
            f"{label:12s}"
            f"{row[0]:10d}"
            f"{row[1]:10d}"
            f"{row[2]:10d}"
        )


def evaluate_generation(
    model,
    raw_split,
    max_samples=None,
    batch_size=32,
):
    model.eval()
    eval_split = select_eval_split(
        raw_split,
        max_samples,
    )
    model_device = next(model.parameters()).device
    predictions, references, examples = [], [], []

    with temporary_padding_side(
        tokenizer,
        "left",
    ), torch.inference_mode():
        for start in range(
            0,
            len(eval_split),
            batch_size,
        ):
            batch = eval_split[
                start:start + batch_size
            ]

            inputs = tokenize_eval_batch(
                batch[TEXT_COLUMN],
                model_device,
            )

            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            generated_texts = tokenizer.batch_decode(
                outputs[
                    :,
                    inputs["input_ids"].shape[1]:,
                ],
                skip_special_tokens=True,
            )

            batch_predictions = [
                normalize_prediction(text)
                for text in generated_texts
            ]
            batch_references = [
                int(label_id)
                for label_id in batch[LABEL_COLUMN]
            ]

            predictions.extend(batch_predictions)
            references.extend(batch_references)

            examples.extend(
                (
                    text,
                    (
                        None
                        if prediction == INVALID_LABEL_ID
                        else CLASS_LABELS[prediction]
                    ),
                    CLASS_LABELS[reference],
                )
                for text, prediction, reference in zip(
                    generated_texts,
                    batch_predictions,
                    batch_references,
                )
            )

    metrics = classification_metrics(
        predictions,
        references,
    )
    metrics.update(
        {
            "valid_output_rate": (
                sum(
                    prediction != INVALID_LABEL_ID
                    for prediction in predictions
                )
                / len(eval_split)
            ),
            "predictions": predictions,
            "references": references,
            "examples": examples,
        }
    )
    return metrics


def evaluate_forced_choice(
    model,
    raw_split,
    max_samples=None,
    batch_size=16,
):
    model.eval()
    eval_split = select_eval_split(
        raw_split,
        max_samples,
    )
    model_device = next(model.parameters()).device
    label_token_ids = LABEL_TOKEN_IDS.to(
        model_device
    )
    predictions, references = [], []

    with temporary_padding_side(
        tokenizer,
        "left",
    ), torch.inference_mode():
        for start in range(
            0,
            len(eval_split),
            batch_size,
        ):
            batch = eval_split[
                start:start + batch_size
            ]

            inputs = tokenize_eval_batch(
                batch[TEXT_COLUMN],
                model_device,
                max_length=MAX_LENGTH - 1,
                add_special_tokens=False,
            )

            label_logits = (
                model(**inputs)
                .logits[:, -1, :]
                .index_select(
                    -1,
                    label_token_ids,
                )
            )

            predictions.extend(
                label_logits.argmax(dim=-1).tolist()
            )
            references.extend(
                int(label_id)
                for label_id in batch[LABEL_COLUMN]
            )

    metrics = classification_metrics(
        predictions,
        references,
    )
    metrics.update(
        {
            "predictions": predictions,
            "references": references,
        }
    )
    return metrics


### 10. Baseline на полном validation split


До создания LoRA adapter измеряются **две baseline-оценки** на полном SST-2 validation split:

- generation-based evaluation — реальная свободная генерация `positive` / `negative`;
- forced-choice evaluation — сравнение log-probabilities двух допустимых labels.

Для обеих оценок считаются accuracy, precision, recall, F1 и confusion matrix.


In [14]:
baseline_generation_metrics = evaluate_generation(
    base_model,
    dataset["validation"],
    max_samples=BASELINE_EVAL_SAMPLES,
    batch_size=GENERATION_BATCH_SIZE,
)

baseline_forced_choice_metrics = evaluate_forced_choice(
    base_model,
    dataset["validation"],
    max_samples=BASELINE_EVAL_SAMPLES,
    batch_size=FORCED_CHOICE_BATCH_SIZE,
)

assert (
    baseline_generation_metrics["total"]
    == len(dataset["validation"])
)

assert (
    baseline_forced_choice_metrics["total"]
    == len(dataset["validation"])
)

print_classification_summary(
    "Baseline — generation-based evaluation",
    baseline_generation_metrics,
)

print_classification_summary(
    "Baseline — forced-choice evaluation",
    baseline_forced_choice_metrics,
)

print("\nGeneration examples:")

for generated, prediction, reference in (
    baseline_generation_metrics["examples"][:10]
):
    print(
        f"generated={generated!r:20s} "
        f"parsed={prediction!r:10s} "
        f"reference={reference}"
    )

RESULTS_TABLE.append(
    {
        "variant": "Исходная модель",
        "generation_accuracy": baseline_generation_metrics["accuracy"],
        "forced_choice_accuracy": baseline_forced_choice_metrics["accuracy"],
        "generation_macro_f1": baseline_generation_metrics["macro_f1"],
        "forced_choice_macro_f1": baseline_forced_choice_metrics["macro_f1"],
        "valid_output_rate": baseline_generation_metrics["valid_output_rate"],
    }
)



Baseline — generation-based evaluation
--------------------------------------
Accuracy:  3.10%
Macro F1:  0.0573
negative precision=0.0000 recall=0.0000 f1=0.0000
positive precision=1.0000 recall=0.0608 f1=0.1146
Valid output rate: 3.10%

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative             0         0       428
positive             0        27       417

Baseline — forced-choice evaluation
-----------------------------------
Accuracy:  51.49%
Macro F1:  0.3502
negative precision=1.0000 recall=0.0117 f1=0.0231
positive precision=0.5121 recall=1.0000 f1=0.6773

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative             5       423         0
positive             0       444         0

Generation examples:
generated='\n\n<think>\nHmm'   parsed=None       reference=positive
generated='\n\n<think>\nHmm'   parsed=None       reference=negative
generated='\n\n<think>\nHmm'   par

### 11. Создание LoRA configuration


Используется стандартная LoRA без quantization и без современных вариантов вроде DoRA, rsLoRA, PiSSA или LoftQ.

Цель этого notebook — сначала изучить классический механизм LoRA.


In [15]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_rslora=False,
)

print(lora_config)


LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


### 12. Создание PeftModel


`get_peft_model()` замораживает base weights и внедряет LoRA matrices в выбранные modules.


In [16]:
model = get_peft_model(
    base_model,
    lora_config,
)

model.config.use_cache = False

total_params, trainable_params = parameter_stats(model)

model.print_trainable_parameters()

print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable share:      {100 * trainable_params / total_params:.6f}%")


trainable params: 16,819,200 || all params: 1,898,644,288 || trainable%: 0.8859

Total parameters:     1,898,644,288
Trainable parameters: 16,819,200
Trainable share:      0.885853%


### 13. Проверка внедрённых LoRA modules


Проверяем не только `requires_grad`, но и сами modules, в которые PEFT действительно добавил `lora_A` и `lora_B`.


In [17]:
lora_module_names = [
    name
    for name, module in model.named_modules()
    if hasattr(module, "lora_A")
    and hasattr(module, "lora_B")
]

lora_target_counts = Counter(
    name.rsplit(".", 1)[-1]
    for name in lora_module_names
)

lora_parameter_counts = defaultdict(int)

for name, parameter in model.named_parameters():
    if "lora_" not in name:
        continue

    target_path = name.split(".lora_", 1)[0]
    target_leaf = target_path.rsplit(".", 1)[-1]

    lora_parameter_counts[target_leaf] += (
        parameter.numel()
    )

print(
    f"LoRA-targeted modules: "
    f"{len(lora_module_names)}"
)

print("\nLoRA module statistics:")
print(
    f"{'module':28s} "
    f"{'instances':>10s} "
    f"{'LoRA params':>16s}"
)
print("-" * 58)

for name in sorted(lora_target_counts):
    print(
        f"{name:28s} "
        f"{lora_target_counts[name]:10d} "
        f"{lora_parameter_counts[name]:16,d}"
    )

print("\nFirst LoRA modules:")

for name in lora_module_names[:40]:
    print(" -", name)

assert lora_module_names, (
    "No LoRA modules were injected."
)


LoRA-targeted modules: 186

LoRA module statistics:
module                        instances      LoRA params
----------------------------------------------------------
down_proj                            24        3,145,728
gate_proj                            24        3,145,728
in_proj_a                            18          594,432
in_proj_b                            18          594,432
in_proj_qkv                          18        2,359,296
in_proj_z                            18        1,179,648
k_proj                                6          245,760
o_proj                                6          393,216
out_proj                             18        1,179,648
q_proj                                6          589,824
up_proj                              24        3,145,728
v_proj                                6          245,760

First LoRA modules:
 - base_model.model.model.layers.0.linear_attn.out_proj
 - base_model.model.model.layers.0.linear_attn.in_proj_qkv
 - base_mode

### 14. Проверка frozen base weights


Trainable parameters должны относиться к LoRA adapter.

Эта ячейка проверяет это автоматически.


In [18]:
trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

unexpected_trainable = [
    name
    for name in trainable_names
    if "lora_" not in name
]

print(f"Trainable tensors: {len(trainable_names)}")

for name in trainable_names[:40]:
    print(" -", name)

if len(trainable_names) > 40:
    print(f"... and {len(trainable_names) - 40} more")

assert not unexpected_trainable, (
    "Unexpected non-LoRA trainable parameters:\n"
    + "\n".join(unexpected_trainable[:20])
)


Trainable tensors: 372
 - base_model.model.model.layers.0.linear_attn.out_proj.lora_A.default.weight
 - base_model.model.model.layers.0.linear_attn.out_proj.lora_B.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_qkv.lora_A.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_qkv.lora_B.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_z.lora_A.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_z.lora_B.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_b.lora_A.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_b.lora_B.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_a.lora_A.default.weight
 - base_model.model.model.layers.0.linear_attn.in_proj_a.lora_B.default.weight
 - base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
 - base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight
 - base_model.model.model.layers.0.mlp.up_p

### 15. Linear LR scheduler и Early Stopping


В этом notebook learning rate schedule и остановка обучения задаются явно.

Используется **Linear LR scheduler**:

```text
warmup
→ learning rate растёт от 0 до LEARNING_RATE
→ после warmup линейно уменьшается к 0
```

При `WARMUP_STEPS = 0.05` первые 5% запланированных optimizer steps используются для warmup. После этого learning rate линейно уменьшается до конца запланированного training run.

Такой schedule особенно полезен для fine-tuning: в начале LoRA adapter получает достаточно крупные обновления, а по мере обучения шаг оптимизации становится меньше.


#### Почему уменьшение learning rate важно


Если learning rate слишком велик, optimizer может двигаться в правильном направлении, но перескакивать через область с меньшим loss.

Условно:

```text
текущая точка
      \
       \        следующий шаг слишком большой
        \      /
         \____/   ← минимум
              \
               X
```

То есть направление gradient может быть правильным, но величина шага слишком велика для того, чтобы остаться в узкой области минимума.

Linear LR scheduler уменьшает размер шага по мере обучения. Поэтому на поздних этапах optimizer получает возможность делать более точные обновления и устойчивее сходиться в области низкого loss.

Scheduler не гарантирует нахождение глобального минимума, но уменьшает риск продолжать делать одинаково крупные шаги на всём training run.


#### Early Stopping


**Early Stopping** следит за validation metric и прекращает обучение, если она перестала улучшаться.

В notebook контролируется:

```python
metric_for_best_model="eval_loss"
greater_is_better=False
```

Используются настройки:

```python
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0
```

`patience=3` означает, что после трёх последовательных evaluation без улучшения `eval_loss` Trainer завершает training.

`threshold=0.0` означает, что любое реальное уменьшение `eval_loss` считается улучшением. Это сделано намеренно: небольшое, но устойчивое движение вниз не должно игнорироваться только из-за заранее заданного порога.


#### Как scheduler и Early Stopping работают вместе


Эти механизмы решают разные задачи:

```text
Linear LR scheduler
→ постепенно уменьшает величину optimizer step

Early Stopping
→ прекращает обучение, когда validation перестаёт улучшаться

load_best_model_at_end=True
→ после завершения восстанавливает лучший сохранённый checkpoint
```

Поэтому, если модель достигла хорошего validation loss, а затем следующие обновления ухудшили результат, финальной остаётся не последняя версия модели, а checkpoint с минимальным наблюдавшимся `eval_loss`.

Early Stopping сам по себе не предотвращает overshooting. Основную роль здесь играет уменьшение learning rate. Early Stopping ограничивает дальнейшее обучение после того, как улучшения прекратились.


#### Что если минимум находится между evaluation points


Validation выполняется каждые `EVAL_STEPS = 200` optimizer steps, и checkpoint сохраняется с той же частотой.

Это означает, что Trainer выбирает лучший результат только среди реально измеренных validation points. Очень короткий минимум, возникший между двумя evaluation, теоретически может быть пропущен.

Если training curve показывает заметные колебания `eval_loss` или признаки того, что модель быстро проходит через область оптимума, первым изменением стоит сделать более частую validation:

```python
EVAL_STEPS = 100
SAVE_STEPS = EVAL_STEPS
```

а не сразу увеличивать `patience`.

Так checkpoints располагаются плотнее по training trajectory, и вероятность пропустить лучший наблюдаемый участок уменьшается. Цена — более частый evaluation и немного большее время обучения.


### 16. TrainingArguments


LoRA использует значительно меньший learning rate, чем Prompt Tuning. В `TrainingArguments` явно задаётся `lr_scheduler_type="linear"`, описанный в предыдущем разделе.

Evaluation и checkpointing выполняются через фиксированное число optimizer steps:

$\text{eval interval} = 200\,\text{steps}$

Для default `smoke`-режима с примерно 1 500 optimizer steps это даёт несколько промежуточных validation measurements вместо всего трёх точек по границам эпох.

Это важно, потому что minimum validation loss может находиться внутри эпохи. Параметры

```python
load_best_model_at_end=True
metric_for_best_model="eval_loss"
greater_is_better=False
```

заставляют `Trainer` после завершения обучения восстановить checkpoint с минимальным validation loss среди сохранённых точек.

Gradient checkpointing и `torch.compile` являются независимыми переключателями:

```text
USE_GRADIENT_CHECKPOINTING = False
USE_TORCH_COMPILE = False
```

Checkpointing стоит включать при дефиците памяти. `torch.compile` имеет смысл тестировать на более длительном training run, где стоимость первоначальной compilation может окупиться.


#### Метрики промежуточной validation


Во время `Trainer` evaluation считаются две дополнительные дешёвые метрики:

- `Label Token Accuracy` — по argmax только на незамаскированных target positions, без отдельного generation-прохода;
- `Perplexity` — из уже вычисленного validation loss.

Для уменьшения объёма данных, передаваемых из evaluation loop в `compute_metrics`, полный tensor logits заменяется на `argmax` через `preprocess_logits_for_metrics`.


In [19]:
def preprocess_logits_for_metrics(
    logits,
    labels,
):
    if isinstance(logits, tuple):
        logits = logits[0]

    return logits.argmax(dim=-1)


def compute_trainer_metrics(eval_prediction):
    predictions = eval_prediction.predictions
    labels = eval_prediction.label_ids

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    shifted_predictions = predictions[:, :-1]
    shifted_labels = labels[:, 1:]

    valid_mask = shifted_labels != -100

    if tokenizer.eos_token_id is not None:
        valid_mask &= (
            shifted_labels
            != tokenizer.eos_token_id
        )

    valid_count = int(valid_mask.sum())

    if valid_count:
        correct_count = int(
            (
                shifted_predictions[valid_mask]
                == shifted_labels[valid_mask]
            ).sum()
        )

        label_token_accuracy = (
            correct_count / valid_count
        )
    else:
        label_token_accuracy = 0.0

    metrics = {
        "label_token_accuracy": (
            label_token_accuracy
        )
    }

    losses = getattr(
        eval_prediction,
        "losses",
        None,
    )

    if losses is not None:
        losses = np.asarray(losses)
        mean_loss = float(losses.mean())

        if np.isfinite(mean_loss):
            metrics["perplexity"] = float(
                np.exp(mean_loss)
            )

    return metrics


In [20]:
use_bf16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

use_fp16 = (
    device.type == "cuda"
    and not use_bf16
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    bf16=use_bf16,
    fp16=use_fp16,
    tf32=True if device.type == "cuda" else None,

    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs=(
        {"use_reentrant": False}
        if USE_GRADIENT_CHECKPOINTING
        else None
    ),

    torch_compile=USE_TORCH_COMPILE,

    optim=(
        "adamw_torch_fused"
        if device.type == "cuda"
        else "adamw_torch"
    ),

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=(device.type == "cuda"),

    include_for_metrics=["loss"],

    report_to="none",

    push_to_hub=False,
)

print("Run mode:", RUN_MODE)
print("Training examples:", len(dataset["train"]))
print("Eval interval:", EVAL_STEPS, "optimizer steps")
print("LR scheduler:", LR_SCHEDULER_TYPE)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Early stopping threshold:", EARLY_STOPPING_THRESHOLD)
print("BF16:", use_bf16)
print("FP16:", use_fp16)
print("Gradient checkpointing:", USE_GRADIENT_CHECKPOINTING)
print("torch.compile:", USE_TORCH_COMPILE)
print("Checkpoint output:", CHECKPOINT_DIR)


Run mode: smoke
Training examples: 8000
Eval interval: 100 optimizer steps
LR scheduler: linear
Early stopping patience: 3
Early stopping threshold: 0.0
BF16: True
FP16: False
Gradient checkpointing: False
torch.compile: False
Checkpoint output: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-lora/checkpoints


### 17. Trainer


Trainer получает обычный `PeftModel`, но optimizer обновляет только параметры с `requires_grad=True` — LoRA matrices.


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_trainer_metrics,
    preprocess_logits_for_metrics=(
        preprocess_logits_for_metrics
    ),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=(
                EARLY_STOPPING_PATIENCE
            ),
            early_stopping_threshold=(
                EARLY_STOPPING_THRESHOLD
            ),
        )
    ],
)


### 18. Обучение


Для продолжения прерванного запуска можно использовать:

```python
trainer.train(resume_from_checkpoint=True)
```


#### Примечание о сообщении про PAD/BOS/EOS tokens


При запуске `Trainer` может появиться информационное сообщение:

```text
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values.
```

Для этого notebook такое сообщение нормально и не означает ошибку обучения.

Причина в том, что tokenizer и конфигурация модели могут изначально содержать разные значения special tokens. В частности, если у tokenizer отсутствует отдельный `pad_token`, выше в notebook выполняется:

```python
tokenizer.pad_token = tokenizer.eos_token
```

То есть для padding используется уже существующий EOS token. После этого `Trainer` проверяет special-token IDs tokenizer и синхронизирует `model.config` и `generation_config` с фактическими значениями tokenizer.

Сообщение перечисляет `PAD/BOS/EOS` как категории special tokens в общей проверке; это не означает, что notebook обязательно изменил все три токена.

Для loss это не создаёт проблемы:

- padding positions исключаются через `attention_mask`;
- padding positions в `labels` заменяются на `-100`;
- PyTorch loss игнорирует позиции с `-100`;
- новые веса словаря из-за присвоения `pad_token = eos_token` не создаются, потому что используется уже существующий token.

Поэтому такое сообщение можно воспринимать как уведомление о согласовании конфигураций tokenizer и модели, а не как warning о некорректном training setup.


In [22]:
train_result = trainer.train()

trainer.log_metrics(
    "train",
    train_result.metrics,
)

trainer.save_metrics(
    "train",
    train_result.metrics,
)

trainer.save_state()

training_performance = dict(
    train_result.metrics
)

training_history = list(
    trainer.state.log_history
)

print("\nTraining performance:")

for key in (
    "train_runtime",
    "train_samples_per_second",
    "train_steps_per_second",
    "train_loss",
):
    if key in training_performance:
        print(
            f"{key:28s}: "
            f"{training_performance[key]}"
        )


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Step,Training Loss,Validation Loss,Label Token Accuracy,Perplexity
100,0.124571,0.095194,0.927752,1.099872
200,0.097204,0.096989,0.933486,1.101848
300,0.086169,0.088504,0.941514,1.092539
400,0.126543,0.089798,0.947248,1.093954
500,0.109725,0.076928,0.948394,1.079964
600,0.070630,0.093412,0.946101,1.097914
700,0.071700,0.085711,0.951835,1.089491
800,0.057862,0.094719,0.950688,1.099350


***** train metrics *****
  epoch                    =        1.6
  total_flos               =  4848042GF
  train_loss               =     0.2219
  train_runtime            = 0:15:56.56
  train_samples_per_second =      25.09
  train_steps_per_second   =      1.568

Training performance:
train_runtime               : 956.5689
train_samples_per_second    : 25.09
train_steps_per_second      : 1.568
train_loss                  : 0.22192771300673486


### 19. Evaluation loss


Trainer выполняет causal language modeling evaluation на полном validation split.

При каждом evaluation interval в progress-таблицу выводятся:

- `Validation Loss`;
- `Label Token Accuracy`;
- `Perplexity`.

`Label Token Accuracy` рассчитывается по target-token argmax без отдельного generation-прохода, а perplexity получается из validation loss.


In [23]:
eval_metrics = trainer.evaluate()

final_perplexity = eval_metrics.get(
    "eval_perplexity"
)

if final_perplexity is not None:
    final_perplexity = float(
        final_perplexity
    )

trainer.log_metrics(
    "eval",
    eval_metrics,
)

trainer.save_metrics(
    "eval",
    eval_metrics,
)

eval_metrics


Training Loss,Validation Loss,Step,Label Token Accuracy,Perplexity
0.057862,0.076928,800,0.948394,1.079964


***** eval metrics *****
  eval_label_token_accuracy = 0.9484
  eval_loss                 = 0.0769
  eval_perplexity           =   1.08


{'eval_loss': 0.07692798972129822,
 'eval_label_token_accuracy': 0.948394495412844,
 'eval_perplexity': 1.0799643050618708}

#### Training dynamics


`Trainer.state.log_history` содержит историю training loss, validation loss и learning rate.

Все три ряда отображаются на одном интерактивном Plotly-графике:

- `train loss` и `validation loss` используют левую Y-ось;
- `learning rate` использует правую Y-ось;
- общая X-ось показывает epoch.

Две Y-оси нужны потому, что loss и learning rate имеют разные масштабы. Такой вариант позволяет одновременно видеть качество обучения и работу Linear LR scheduler, не занимая место двумя отдельными графиками.


In [24]:
train_loss_points = [
    (
        entry["epoch"],
        entry["loss"],
    )
    for entry in training_history
    if (
        "loss" in entry
        and "eval_loss" not in entry
        and "epoch" in entry
    )
]

eval_loss_points = [
    (
        entry["epoch"],
        entry["eval_loss"],
    )
    for entry in training_history
    if (
        "eval_loss" in entry
        and "epoch" in entry
    )
]

learning_rate_points = [
    (
        entry["epoch"],
        entry["learning_rate"],
    )
    for entry in training_history
    if (
        "learning_rate" in entry
        and "epoch" in entry
    )
]

training_fig = go.Figure()

if train_loss_points:
    train_epochs, train_losses = zip(
        *train_loss_points
    )

    training_fig.add_trace(
        go.Scatter(
            x=train_epochs,
            y=train_losses,
            mode="lines+markers",
            name="train loss",
            yaxis="y",
        )
    )

if eval_loss_points:
    eval_epochs, eval_losses = zip(
        *eval_loss_points
    )

    training_fig.add_trace(
        go.Scatter(
            x=eval_epochs,
            y=eval_losses,
            mode="lines+markers",
            name="validation loss",
            yaxis="y",
        )
    )

if learning_rate_points:
    lr_epochs, learning_rates = zip(
        *learning_rate_points
    )

    training_fig.add_trace(
        go.Scatter(
            x=lr_epochs,
            y=learning_rates,
            mode="lines",
            name="learning rate",
            yaxis="y2",
        )
    )

training_fig.update_layout(
    title="Training dynamics",
    height=430,
    xaxis={
        "title": "Epoch",
    },
    yaxis={
        "title": "Loss",
    },
    yaxis2={
        "title": "Learning rate",
        "overlaying": "y",
        "side": "right",
        "tickformat": ".1e",
        "showgrid": False,
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "left",
        "x": 0,
    },
    margin={
        "l": 70,
        "r": 90,
        "t": 90,
        "b": 60,
    },
)

training_fig.show()


### 20. Accuracy после LoRA


После training повторяются **оба** baseline evaluation режима на том же полном validation split:

- свободная generation-based классификация;
- forced-choice классификация по conditional log-probabilities.

Это позволяет увидеть отдельно изменение task knowledge и изменение output behavior.


In [25]:
model = trainer.model
model.config.use_cache = True

final_generation_metrics = evaluate_generation(
    model,
    dataset["validation"],
    max_samples=FINAL_EVAL_SAMPLES,
    batch_size=GENERATION_BATCH_SIZE,
)

final_forced_choice_metrics = evaluate_forced_choice(
    model,
    dataset["validation"],
    max_samples=FINAL_EVAL_SAMPLES,
    batch_size=FORCED_CHOICE_BATCH_SIZE,
)

assert (
    final_generation_metrics["total"]
    == len(dataset["validation"])
)

assert (
    final_forced_choice_metrics["total"]
    == len(dataset["validation"])
)

print_classification_summary(
    "LoRA — generation-based evaluation",
    final_generation_metrics,
)

print_classification_summary(
    "LoRA — forced-choice evaluation",
    final_forced_choice_metrics,
)

print("\nComparison:")

print(
    "Generation accuracy: "
    f"{baseline_generation_metrics['accuracy']:.2%}"
    " → "
    f"{final_generation_metrics['accuracy']:.2%}"
)

print(
    "Forced-choice accuracy: "
    f"{baseline_forced_choice_metrics['accuracy']:.2%}"
    " → "
    f"{final_forced_choice_metrics['accuracy']:.2%}"
)

print(
    "Generation Macro F1: "
    f"{baseline_generation_metrics['macro_f1']:.4f}"
    " → "
    f"{final_generation_metrics['macro_f1']:.4f}"
)

print(
    "Forced-choice Macro F1: "
    f"{baseline_forced_choice_metrics['macro_f1']:.4f}"
    " → "
    f"{final_forced_choice_metrics['macro_f1']:.4f}"
)

print(
    "Valid output rate: "
    f"{baseline_generation_metrics['valid_output_rate']:.2%}"
    " → "
    f"{final_generation_metrics['valid_output_rate']:.2%}"
)

print("\nGeneration examples:")

for generated, prediction, reference in (
    final_generation_metrics["examples"][:10]
):
    print(
        f"generated={generated!r:20s} "
        f"parsed={prediction!r:10s} "
        f"reference={reference}"
    )

artifact_expected_generation = (
    final_generation_metrics[
        "predictions"
    ][:ARTIFACT_RELOAD_SAMPLES]
)

artifact_expected_forced_choice = (
    final_forced_choice_metrics[
        "predictions"
    ][:ARTIFACT_RELOAD_SAMPLES]
)

RESULTS_TABLE.append(
    {
        "variant": "LoRA",
        "generation_accuracy": final_generation_metrics["accuracy"],
        "forced_choice_accuracy": final_forced_choice_metrics["accuracy"],
        "generation_macro_f1": final_generation_metrics["macro_f1"],
        "forced_choice_macro_f1": final_forced_choice_metrics["macro_f1"],
        "valid_output_rate": final_generation_metrics["valid_output_rate"],
    }
)



LoRA — generation-based evaluation
----------------------------------
Accuracy:  94.95%
Macro F1:  0.9495
negative precision=0.9528 recall=0.9439 f1=0.9484
positive precision=0.9464 recall=0.9550 f1=0.9507
Valid output rate: 100.00%

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           404        24         0
positive            20       424         0

LoRA — forced-choice evaluation
-------------------------------
Accuracy:  94.84%
Macro F1:  0.9484
negative precision=0.9485 recall=0.9463 f1=0.9474
positive precision=0.9483 recall=0.9505 f1=0.9494

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           405        23         0
positive            22       422         0

Comparison:
Generation accuracy: 3.10% → 94.95%
Forced-choice accuracy: 51.49% → 94.84%
Generation Macro F1: 0.0573 → 0.9495
Forced-choice Macro F1: 0.3502 → 0.9484
Valid output rate: 3.10% → 100.00%

Gen

In [26]:
del trainer
del train_result
del eval_metrics

clear_device_memory()


### 21. Сохранение LoRA adapter


В `OUTPUT_DIR` сохраняются LoRA weights и tokenizer.

Base model не копируется: она продолжает использоваться из общего Hugging Face cache.


In [27]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved LoRA adapter to: {OUTPUT_DIR}")


Saved LoRA adapter to: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-lora


#### Reproducibility metadata


Помимо seed и package versions notebook фиксирует конкретную revision модели, revision dataset repository и fingerprints реально использованных train/validation splits.

Это особенно важно при использовании `revision=None`, потому что такой запуск фактически использует текущее состояние ветки `main`.


In [28]:
import json as json_module

REPRODUCIBILITY_PATH = (
    OUTPUT_DIR / "reproducibility.json"
)

reproducibility_data = {
    "seed": SEED,
    "run_mode": RUN_MODE,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_steps": WARMUP_STEPS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "model_id": MODEL_ID,
    "model_revision_requested": MODEL_REVISION,
    "model_revision_resolved": resolved_model_revision,
    "dataset_id": DATASET_ID,
    "dataset_revision_requested": DATASET_REVISION,
    "dataset_revision_resolved": resolved_dataset_revision,
    "train_fingerprint": dataset_train_fingerprint,
    "validation_fingerprint": dataset_validation_fingerprint,
    "train_examples": len(dataset["train"]),
    "validation_examples": len(dataset["validation"]),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": LORA_TARGET_MODULES,
    "gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "torch_compile": USE_TORCH_COMPILE,
}

REPRODUCIBILITY_PATH.write_text(
    json_module.dumps(
        reproducibility_data,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    f"Saved reproducibility metadata: "
    f"{REPRODUCIBILITY_PATH}"
)


Saved reproducibility metadata: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-lora/reproducibility.json


### 22. Размер LoRA adapter


Размер считается только по adapter files, без Trainer checkpoints.


In [29]:
adapter_files = [
    OUTPUT_DIR / "adapter_model.safetensors",
    OUTPUT_DIR / "adapter_config.json",
]

adapter_size_bytes = sum(
    path.stat().st_size
    for path in adapter_files
    if path.exists()
)

print(f"LoRA adapter size:      {adapter_size_bytes / 1024**2:.3f} MiB")
print(f"Base parameter memory:  {parameter_memory_gib(base_model):.2f} GiB")

print("\nFinal export files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(
            f"{file.name:32s} "
            f"{file.stat().st_size / 1024:.1f} KiB"
        )


LoRA adapter size:      64.209 MiB
Base parameter memory:  3.57 GiB

Final export files:
README.md                        5.1 KiB
adapter_config.json              1.2 KiB
adapter_model.safetensors        65749.1 KiB
chat_template.jinja              7.6 KiB
reproducibility.json             0.8 KiB
tokenizer.json                   19521.1 KiB
tokenizer_config.json            1.1 KiB


In [30]:
del model
del base_model

clear_device_memory()

### 23. Проверка сохранённого adapter


Сохранённый LoRA adapter загружается заново поверх чистой base model.

Проверка запускается по умолчанию на нескольких фиксированных validation examples и сравнивает:

- generation predictions до сохранения и после reload;
- forced-choice predictions до сохранения и после reload.

Так проверяется именно сохранённый артефакт, а не состояние модели, оставшееся после training.


In [31]:
if RUN_ARTIFACT_RELOAD_TEST:
    adapter_config = PeftConfig.from_pretrained(
        OUTPUT_DIR
    )

    reloaded_base_model = (
        AutoModelForCausalLM.from_pretrained(
            adapter_config.base_model_name_or_path,
            revision=MODEL_REVISION,
            dtype=model_dtype,
        )
    )

    reloaded_model = PeftModel.from_pretrained(
        reloaded_base_model,
        OUTPUT_DIR,
    )

    reloaded_model = reloaded_model.to(device)
    reloaded_model.eval()

    reload_generation_metrics = (
        evaluate_generation(
            reloaded_model,
            dataset["validation"],
            max_samples=ARTIFACT_RELOAD_SAMPLES,
            batch_size=ARTIFACT_RELOAD_SAMPLES,
        )
    )

    reload_forced_choice_metrics = (
        evaluate_forced_choice(
            reloaded_model,
            dataset["validation"],
            max_samples=ARTIFACT_RELOAD_SAMPLES,
            batch_size=ARTIFACT_RELOAD_SAMPLES,
        )
    )

    assert (
        reload_generation_metrics["predictions"]
        == artifact_expected_generation
    ), (
        "Generation predictions changed "
        "after adapter reload."
    )

    assert (
        reload_forced_choice_metrics["predictions"]
        == artifact_expected_forced_choice
    ), (
        "Forced-choice predictions changed "
        "after adapter reload."
    )

    print(
        "Adapter reload smoke test: PASSED"
    )
    print(
        f"Checked examples: "
        f"{ARTIFACT_RELOAD_SAMPLES}"
    )

    del reload_generation_metrics
    del reload_forced_choice_metrics
    del reloaded_model
    del reloaded_base_model
    del adapter_config

    clear_device_memory()
else:
    print(
        "RUN_ARTIFACT_RELOAD_TEST=False — "
        "reload test skipped."
    )


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Adapter reload smoke test: PASSED
Checked examples: 8


### 24. Merge LoRA в base model


LoRA adapter можно слить с исходными weights через `merge_and_unload()`.

Это полезно для deployment, если отдельный adapter больше не нужен.

Ячейка выключена по умолчанию, потому что merged model занимает размер полной Qwen3.5 и требует отдельного сохранения.


In [32]:
MERGE_ADAPTER = False

if MERGE_ADAPTER:
    merge_base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=model_dtype,
    )

    merge_base_model = merge_base_model.to(device)

    merge_model = PeftModel.from_pretrained(
        merge_base_model,
        OUTPUT_DIR,
    )

    merged_model = merge_model.merge_and_unload()

    print(type(merged_model).__name__)
    print("LoRA weights merged into the base model.")

    del merged_model
    del merge_model
    del merge_base_model

    clear_device_memory()
else:
    print(
        "Set MERGE_ADAPTER = True "
        "to test merge_and_unload()."
    )


Set MERGE_ADAPTER = True to test merge_and_unload().


### 25. Создание Hugging Face Model Card


Model Card создаётся как компактный `README.md` для опубликованного LoRA adapter.

В карточке остаются только:

- краткое описание;
- базовые сведения о модели и LoRA;
- основные результаты;
- минимальный пример запуска через Transformers + PEFT;
- короткие ограничения.


In [33]:
MODEL_CARD_PATH = OUTPUT_DIR / "README.md"

model_display_name = HUB_MODEL_ID.split("/")[-1]

evaluation_is_full_validation = (
    final_generation_metrics["total"]
    == len(raw_dataset["validation"])
)

evaluation_scope = (
    "full SST-2 validation split"
    if evaluation_is_full_validation
    else (
        f"{final_generation_metrics['total']} "
        "SST-2 validation examples"
    )
)

perplexity_text = (
    f"{final_perplexity:.4f}"
    if final_perplexity is not None
    else "n/a"
)

card_text = f"""---
base_model: {MODEL_ID}
library_name: peft
pipeline_tag: text-generation
datasets:
- {DATASET_ID}
language:
- en
license: apache-2.0
tags:
- peft
- lora
- qwen3.5
- sentiment-analysis
---

# {model_display_name}

LoRA adapter for
[`{MODEL_ID}`](https://huggingface.co/{MODEL_ID}),
fine-tuned on
[`{DATASET_ID}`](https://huggingface.co/datasets/{DATASET_ID})
for binary sentiment classification.

The adapter predicts `positive` or `negative`.

## Model Details

| Property | Value |
|---|---|
| Base model | `{MODEL_ID}` |
| Method | LoRA |
| Dataset | `{DATASET_ID}` |
| Task | Sentiment classification |
| Labels | `negative`, `positive` |
| LoRA rank | {LORA_R} |
| LoRA alpha | {LORA_ALPHA} |
| LoRA dropout | {LORA_DROPOUT} |
| Target modules | `{LORA_TARGET_MODULES}` |
| Adapter size | {adapter_size_bytes / 1024**2:.2f} MiB |

## Evaluation

Evaluation scope: **{evaluation_scope}**.

| Metric | Base model | LoRA |
|---|---:|---:|
| Generation accuracy | {baseline_generation_metrics["accuracy"]:.2%} | **{final_generation_metrics["accuracy"]:.2%}** |
| Forced-choice accuracy | {baseline_forced_choice_metrics["accuracy"]:.2%} | **{final_forced_choice_metrics["accuracy"]:.2%}** |
| Generation Macro F1 | {baseline_generation_metrics["macro_f1"]:.4f} | **{final_generation_metrics["macro_f1"]:.4f}** |
| Forced-choice Macro F1 | {baseline_forced_choice_metrics["macro_f1"]:.4f} | **{final_forced_choice_metrics["macro_f1"]:.4f}** |
| Perplexity | — | {perplexity_text} |

## Usage

```python
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "{MODEL_ID}"
ADAPTER_ID = "{HUB_MODEL_ID}"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

review = "a wonderfully acted and moving story"

prompt = (
    "{VISIBLE_INSTRUCTION}\\n"
    f"Review: {{review}}\\n"
    "Sentiment:"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens={MAX_NEW_TOKENS},
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[
    :,
    inputs["input_ids"].shape[1]:,
]

prediction = tokenizer.decode(
    generated[0],
    skip_special_tokens=True,
).strip()

print(prediction)
```

## Limitations

- Designed for English SST-2 sentiment classification.
- Requires the documented prompt format and the base model `{MODEL_ID}`.
"""

MODEL_CARD_PATH.write_text(
    card_text,
    encoding="utf-8",
)

print(f"Saved Model Card: {MODEL_CARD_PATH}")


Saved Model Card: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-lora/README.md


### 26. Публикация в Hugging Face Hub


Push отключён по умолчанию.

Перед публикацией выполните `hf auth login` и установите `PUSH_TO_HUB=True`.

На Hub отправляются LoRA adapter, tokenizer, `README.md` и `reproducibility.json`. Trainer checkpoints исключаются.


In [34]:
if PUSH_TO_HUB:
    create_repo(
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        exist_ok=True,
    )

    api = HfApi()

    commit_info = api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        ignore_patterns=[
            "checkpoints/**",
            "checkpoint-*/**",
            "runs/**",
            "*.pt",
            "*.pth",
        ],
        commit_message=(
            "Upload Qwen3.5-2B SST-2 LoRA adapter "
            "and model card"
        ),
    )

    print(
        f"Published: "
        f"https://huggingface.co/{HUB_MODEL_ID}"
    )

    print(f"Commit:    {commit_info.commit_url}")
else:
    print(
        "PUSH_TO_HUB=False - nothing was uploaded."
    )


PUSH_TO_HUB=False - nothing was uploaded.


## Результаты


### Итоговая сравнительная таблица

In [35]:
assert [row["variant"] for row in RESULTS_TABLE] == [
    "Исходная модель",
    "LoRA",
]

print(
    f"{'Вариант':18s}"
    f"{'Generation Accuracy':>22s}"
    f"{'Forced-Choice Accuracy':>24s}"
    f"{'Generation Macro F1':>22s}"
    f"{'Forced-Choice Macro F1':>24s}"
    f"{'Valid Output Rate':>20s}"
)

print("-" * 130)

for row in RESULTS_TABLE:
    print(
        f"{row['variant']:18s}"
        f"{row['generation_accuracy']:22.2%}"
        f"{row['forced_choice_accuracy']:24.2%}"
        f"{row['generation_macro_f1']:22.4f}"
        f"{row['forced_choice_macro_f1']:24.4f}"
        f"{row['valid_output_rate']:20.2%}"
    )


Вариант              Generation Accuracy  Forced-Choice Accuracy   Generation Macro F1  Forced-Choice Macro F1   Valid Output Rate
----------------------------------------------------------------------------------------------------------------------------------
Исходная модель                    3.10%                  51.49%                0.0573                  0.3502               3.10%
LoRA                              94.95%                  94.84%                0.9495                  0.9484             100.00%


Сравнительная таблица показывает резкое улучшение качества после LoRA сразу в обоих режимах оценки.

У исходной модели **Generation Accuracy составляет 3.10%**, а **Valid Output Rate — 3.10%**. Это означает, что при свободной генерации модель почти всегда нарушает требуемый формат ответа: вместо `positive` или `negative` она продолжает промпт произвольным текстом. Поэтому низкую Generation Accuracy нельзя интерпретировать как прямую оценку способности базовой модели распознавать тональность.

**Forced-Choice Accuracy исходной модели составляет 51.49%**, а **Forced-Choice Macro F1 — 0.3502**. Результат близок к случайному выбору для бинарной классификации и показывает, что проблема исходной модели не ограничивается только форматом генерации: без адаптации она также слабо разделяет классы SST-2 в выбранной постановке задачи.

После LoRA **Generation Accuracy возрастает до 94.95%**, а **Forced-Choice Accuracy — до 94.84%**. Прирост относительно исходной модели составляет соответственно **+91.85** и **+43.35 процентного пункта**. При этом **Generation Macro F1 достигает 0.9495**, а **Forced-Choice Macro F1 — 0.9484**.

**Valid Output Rate возрастает с 3.10% до 100.00%**, то есть после адаптации модель стабильно соблюдает требуемый формат ответа. Разница между Generation Accuracy и Forced-Choice Accuracy после LoRA составляет всего **0.11 процентного пункта**, а между соответствующими значениями Macro F1 — **0.0011**. Практически одинаковый результат двух независимых режимов оценки показывает, что модель не только научилась различать классы, но и корректно выражать решение в требуемом формате.

Таким образом, в этом эксперименте LoRA одновременно решила две задачи: существенно улучшила качество бинарной классификации SST-2 и сформировала устойчивое поведение при генерации ответа. При этом базовые веса модели остаются замороженными, а результат сохраняется как компактный LoRA-адаптер.

## Источники


- LoRA: https://arxiv.org/abs/2106.09685
- PEFT LoRA conceptual guide: https://huggingface.co/docs/peft/main/conceptual_guides/lora
- PEFT LoRA API: https://huggingface.co/docs/peft/main/package_reference/lora
- PEFT developer guides: https://huggingface.co/docs/peft/main/developer_guides/lora
- Transformers PEFT integration: https://huggingface.co/docs/transformers/peft
- Transformers torch.compile: https://huggingface.co/docs/transformers/perf_torch_compile
- PyTorch torch.compile: https://docs.pytorch.org/docs/stable/generated/torch.compile.html
- Qwen3.5 docs: https://huggingface.co/docs/transformers/model_doc/qwen3_5
- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base
- Stanford SST-2: https://huggingface.co/datasets/stanfordnlp/sst2
